In [1]:
import arcpy
from arcpy import env
import os
import numpy as np
from arcgis import GIS
from arcgis.features import GeoAccessor
from arcgis.features import GeoSeriesAccessor
import pandas as pd

arcpy.env.overwriteOutput = True
arcpy.env.parallelProcessingFactor = "90%"

# show all columns
pd.options.display.max_columns = None

# pd.pivot_table(df, values='a', index='b', columns='c', aggfunc='sum', fill_value=0)
# pd.DataFrame.spatial.from_featureclass(???)  
# df.spatial.to_featureclass(location=???,sanitize_columns=False)  

# gsa = arcgis.features.GeoSeriesAccessor(df['SHAPE'])  
# df['AREA'] = gsa.area  # KNOW YOUR UNITS

In [ ]:
## spatial join
# target_features = ?
# join_features = ?
# output_features = os.path.join(gdb, ?)

# fieldmappings = arcpy.FieldMappings()
# fieldmappings.addTable(target_features)
# fieldmappings.addTable(join_features)

# # variable
# fieldindex = fieldmappings.findFieldMapIndex(?)
# fieldmap = fieldmappings.getFieldMap(fieldindex)
# fieldmap.mergeRule = 'Sum'
# fieldmappings.replaceFieldMap(fieldindex, fieldmap)

# sj = arcpy.SpatialJoin_analysis(target_features, join_features, output_features,'JOIN_ONE_TO_ONE', "KEEP_ALL", fieldmappings, match_option="INTERSECT")
# sj_df = pd.DataFrame.spatial.from_featureclass(sj[0]).copy()

In [ ]:
# fill NA values in Spatially enabled dataframes (ignores SHAPE column)
def fill_na_sedf(df_with_shape_column, fill_value=0):
    if 'SHAPE' in list(df_with_shape_column.columns):
        cols_to_fill = df_with_shape_column.columns.difference(['SHAPE'])
        df_with_shape_column[cols_to_fill] = df_with_shape_column[cols_to_fill].fillna(fill_value)
        return df_with_shape_column
    else:
        raise Exception("Dataframe does not include 'SHAPE' column")

In [2]:
outputs = ['.\\Outputs', "scratch.gdb", 'results.gdb']

if not os.path.exists(outputs[0]):
    os.makedirs(outputs[0])

gdb = os.path.join(outputs[0], outputs[1])
gdb2 = os.path.join(outputs[0], outputs[2])

if not arcpy.Exists(gdb):
    arcpy.CreateFileGDB_management(outputs[0], outputs[1])

if not arcpy.Exists(gdb2):
    arcpy.CreateFileGDB_management(outputs[0], outputs[2])

In [19]:
parcel_eq = pd.read_csv(r"E:\Tasks\REMM-Manage-Base-Year-Data-2023\Inputs\Tables\Parcel_EQ.csv")
pj = pd.read_csv(r"E:\Tasks\REMM-Manage-Base-Year-Data-2023\Inputs\Tables\pipelineJobs.csv")

In [23]:
pj_eq = pj.merge(parcel_eq, left_on='building_id', right_on='parcel_id_OLD', how='left')
pj_eq.head()

,building_id,year,jobs,jobs_sector,dev_type,sector_id,parcel_id_NEW,parcel_id_OLD
0,766623,2023,150,government,add additional,3,69247.0,766623
1,766623,2024,110,manufacturing,add additional,5,69247.0,766623
2,766623,2025,122,government,add additional,3,69247.0,766623
3,766623,2025,89,manufacturing,add additional,5,69247.0,766623
4,766623,2025,109,office,add additional,6,69247.0,766623


In [24]:
pj_eq['building_id'] = pj_eq['parcel_id_NEW']
del pj_eq['parcel_id_NEW']
del pj_eq['parcel_id_OLD']

In [25]:
pj_eq.to_csv(os.path.join(outputs[0], 'pipeline_jobs_20260125.csv'), index=False)